# Module 1 Homework: Docker & SQL (Cohort 2026)

This notebook contains the solutions for `cohorts/2026/01-docker-terraform/homework.md`.


## Question 1. Understanding Docker images

Run docker with the `python:3.13` image.

**Answer:** `pip 25.3` (option **25.3**)


In [4]:
# Shell commands in a Python notebook need a leading `!` (or use a `%%bash` cell).
!docker run --rm python:3.13 python -m pip --version


pip 25.3 from /usr/local/lib/python3.13/site-packages/pip (python 3.13)


## Question 2. Understanding Docker networking and docker-compose

Within the `docker-compose.yaml` network, `pgadmin` should connect to Postgres using the service/container DNS name and the container port.

**Answer:** `db:5432` (option **db:5432**)

Note: `postgres:5432` can also work in many setups because `container_name: postgres` sets the container name, but `db:5432` is the canonical Compose answer.


## Prepare the Data

This homework uses:
- `green_tripdata_2025-11.parquet`
- `taxi_zone_lookup.csv`

The cells below assume both files are under `cohorts/2026/01-docker-terraform/data/`.


In [6]:
# If you don't have duckdb installed in your notebook environment, uncomment:
# %pip install duckdb

from pathlib import Path
import duckdb

DATA_DIR = Path("cohorts/2026/01-docker-terraform/data")
TRIPS_PATH = DATA_DIR / "green_tripdata_2025-11.parquet"
ZONES_PATH = DATA_DIR / "taxi_zone_lookup.csv"

TRIPS_PATH, ZONES_PATH


(PosixPath('cohorts/2026/01-docker-terraform/data/green_tripdata_2025-11.parquet'),
 PosixPath('cohorts/2026/01-docker-terraform/data/taxi_zone_lookup.csv'))

In [7]:
# Optional: download the trip parquet if missing.
# (Enable only if you have network access from the notebook environment.)

import urllib.request

TRIPS_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-11.parquet"

if not TRIPS_PATH.exists():
    TRIPS_PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(TRIPS_URL, TRIPS_PATH)
    print("Downloaded:", TRIPS_PATH)
else:
    print("Already exists:", TRIPS_PATH)


Downloaded: cohorts/2026/01-docker-terraform/data/green_tripdata_2025-11.parquet


In [18]:
from pathlib import Path

con = duckdb.connect()

def _find_repo_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / ".git").exists() or (p / "cohorts").exists():
            return p
    return start

def _ensure_views() -> None:
    repo_root = _find_repo_root(Path.cwd())

    trips_path_obj = TRIPS_PATH if "TRIPS_PATH" in globals() else (repo_root / "cohorts/2026/01-docker-terraform/data/green_tripdata_2025-11.parquet")
    zones_path_obj = ZONES_PATH if "ZONES_PATH" in globals() else (repo_root / "cohorts/2026/01-docker-terraform/data/taxi_zone_lookup.csv")

    trips_path_obj = Path(trips_path_obj)
    zones_path_obj = Path(zones_path_obj)

    if not zones_path_obj.exists():
        fallback = repo_root / "04-analytics-engineering/taxi_rides_ny/seeds/taxi_zone_lookup.csv"
        if fallback.exists():
            zones_path_obj = fallback

    if not trips_path_obj.exists():
        raise FileNotFoundError(f"Trips parquet not found: {trips_path_obj}")
    if not zones_path_obj.exists():
        raise FileNotFoundError(f"Zones CSV not found: {zones_path_obj}")

    # DuckDB doesn't support parameter binding for DDL statements like `CREATE VIEW ...`.
    trips_path = trips_path_obj.as_posix().replace("'", "''")
    zones_path = zones_path_obj.as_posix().replace("'", "''")

    con.execute(f"CREATE OR REPLACE VIEW trips AS SELECT * FROM read_parquet('{trips_path}')")
    con.execute(f"CREATE OR REPLACE VIEW zones AS SELECT * FROM read_csv_auto('{zones_path}', header=True)")

_ensure_views()
con.execute("SELECT COUNT(*) AS rows FROM trips").fetchone()


FileNotFoundError: Zones CSV not found: cohorts/2026/01-docker-terraform/data/taxi_zone_lookup.csv

## Question 3. Counting short trips

For trips in November 2025, how many trips had `trip_distance <= 1`?

**Answer:** `8,007` (option **8,007**)


In [ ]:
q3 = con.execute(
    """
    SELECT COUNT(*)
    FROM trips
    WHERE lpep_pickup_datetime >= TIMESTAMP '2025-11-01'
      AND lpep_pickup_datetime <  TIMESTAMP '2025-12-01'
      AND trip_distance <= 1
    """
).fetchone()[0]
q3


8007

## Question 4. Longest trip for each day

Which pickup day had the longest trip distance (consider only `trip_distance < 100`)?

**Answer:** `2025-11-14` (option **2025-11-14**)


In [ ]:
q4 = con.execute(
    """
    SELECT CAST(lpep_pickup_datetime AS DATE) AS pickup_date
    FROM trips
    WHERE lpep_pickup_datetime >= TIMESTAMP '2025-11-01'
      AND lpep_pickup_datetime <  TIMESTAMP '2025-12-01'
      AND trip_distance < 100
    ORDER BY trip_distance DESC
    LIMIT 1
    """
).fetchone()[0]
q4


datetime.date(2025, 11, 14)

## Question 5. Biggest pickup zone

Which pickup zone had the largest `total_amount` sum on November 18th, 2025?

**Answer:** `East Harlem North` (option **East Harlem North**)


In [ ]:
import duckdb

if "con" not in globals():
    con = duckdb.connect()

if "_ensure_views" not in globals():
    raise RuntimeError("Run the setup cell above (it creates the DuckDB views `trips` and `zones`) and then re-run this cell.")

_ensure_views()

q5 = con.execute(
    """
    SELECT z.zone, SUM(t.total_amount) AS total_amount_sum
    FROM trips t
    JOIN zones z ON z.locationid = t.PULocationID
    WHERE CAST(t.lpep_pickup_datetime AS DATE) = DATE '2025-11-18'
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 1
    """
).fetchone()
q5


FileNotFoundError: Zones CSV not found: cohorts/2026/01-docker-terraform/data/taxi_zone_lookup.csv

## Question 6. Largest tip

For passengers picked up in the zone named **East Harlem North** in November 2025, which dropoff zone had the largest tip?

This dataset uses the column name `tip_amount`.

**Answer:** `Yorkville West` (option **Yorkville West**)


In [ ]:
import duckdb

if "con" not in globals():
    con = duckdb.connect()

if "_ensure_views" not in globals():
    raise RuntimeError("Run the setup cell above (it creates the DuckDB views `trips` and `zones`) and then re-run this cell.")

_ensure_views()

q6 = con.execute(
    """
    SELECT zd.zone, MAX(t.tip_amount) AS max_tip
    FROM trips t
    JOIN zones zp ON zp.locationid = t.PULocationID
    JOIN zones zd ON zd.locationid = t.DOLocationID
    WHERE t.lpep_pickup_datetime >= TIMESTAMP '2025-11-01'
      AND t.lpep_pickup_datetime <  TIMESTAMP '2025-12-01'
      AND zp.zone = 'East Harlem North'
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 1
    """
).fetchone()
q6


FileNotFoundError: Zones CSV not found: cohorts/2026/01-docker-terraform/data/taxi_zone_lookup.csv

## Question 7. Terraform Workflow

Workflow for:
1) Downloading provider plugins / setting up backend,
2) Generating proposed changes and auto-executing,
3) Removing all managed resources.

**Answer:** `terraform init`, `terraform apply -auto-approve`, `terraform destroy` (option **terraform init, terraform apply -auto-approve, terraform destroy**)
